In [29]:
import torch                                         #import Pytorch library(this will have tensors, gpu support,math for neural networks).
import torch.nn as nn                                #torch.nn will import to get convolution,relu,crossentropy will br there in it.
import torch.optim as optim                          # here wew are importing optimisers.
from torchvision import datasets, transforms         #datasets will read class wise folders.
                                                    # transforms will preprosses the images like resizing,converting to tensor(higher dim array of numbers) ,normaliztion
from torch.utils.data import DataLoader, random_split# Dataloder is used to load the the in batches
                                                      # randomsplit , this will split the data into traing and testing
                  


In [30]:
print("hi")

hi


In [31]:
data_dir = r"C:\ml\simp data"

In [ ]:
transform = transforms.Compose([   # this used to preprocess the image (one by one)
    transforms.Resize((64, 64)),   # resizing will caues loss of image but keeping all images in one size will increac computation
    transforms.ToTensor()         #this will convert the imagpixels to tensor formate
])

In [33]:
dataset = datasets.ImageFolder(root=data_dir, transform=transform) # this will load the images from our folder and gives the labels for the classes

print("Classes:", dataset.classes)
print("Total Images:", len(dataset))

Classes: ['abraham_grampa_simpson', 'agnes_skinner', 'apu_nahasapeemapetilon', 'barney_gumble', 'bart_simpson']
Total Images: 3026


In [34]:
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = random_split(dataset, [train_size, test_size])  # thsi will ensure that traning and testing will get mixed images

print("Training Images:", len(train_dataset))
print("Testing Images:", len(test_dataset))

Training Images: 2420
Testing Images: 606


In [66]:
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True) # gives the data to the model in small b atches insted all at once.amd for traning we
                                                                     # will shuffel the images for every epoch.
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False)

In [67]:
class SimpleCNN(nn.Module):                                   #if we dony use class then evrthing becoms messy and pytorch will not track the layers properly.
                                                             # optimiser may not know which parameter belog to the model
                                                             #  nn.module , this is a built in parent class for all neural networks .this already knows
                                                            #               to handel layers, parameters ,and etc .taht why we are making our class as subclass of 
                                                            #               this parent class.


    def __init__(self):                                        #constructer(used to define layers)
        super(SimpleCNN, self).__init__()                      # super calls parent class constructer .so here we are initialising the pytorch neural network
                                                               #first and then we are our cnn .
                                                               #self
                                                               # this will reffer to the class object(model objct).if we dont use self then the all the 
                                                               # variabes will become local to the constructer and class cant see it once after the constucter.
                                                               #

        self.conv1 = nn.Conv2d(3, 16, 3,stride=1, padding=1)# inchsnnle,no.of filters,filtersize,padding 
                                                   #dif filters
        self.pool = nn.MaxPool2d(2, 2)
        self.relu = nn.ReLU()

        self.conv2 = nn.Conv2d(16, 32, 3,stride=1, padding=1)

        self.fc1 = nn.Linear(32 * 16 * 16, 64)
        self.fc2 = nn.Linear(64, 5)   # 5 classes

    def forward(self, x):                                      #methood
                                                               #here we defining rhe flow .
                                                               # we cant write every thing inside it as it will create a new layer every time.that is not
                                                               # not learning.

        x = self.pool(self.relu(self.conv1(x)))                # x is nothing but batch of images
        x = self.pool(self.relu(self.conv2(x)))

        x = x.view(x.size(0), -1)                             # we are falttening the data.(batch_size,remaining size)

        x = self.relu(self.fc1(x))
        x = self.fc2(x)

        return x                                              # this will return the scores of the 5 classes of each batch

In [68]:
model = SimpleCNN()
print(model)

SimpleCNN(
  (conv1): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (relu): ReLU()
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (fc1): Linear(in_features=8192, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=5, bias=True)
)


In [69]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)#update weights after 1 mini batch.like in normal sgd we will update weights after one image but in Pytorch
                                                  # the weights are updated after one mini batch.

In [70]:
num_epochs = 10

for epoch in range(num_epochs):
    model.train()                  #this sets the model into traning mode so that layers behavr correctly .and here we will randomly turn off some 
                                   #of the neurons to prevent over fitting but in testing this is off dropout randomly.and batch norm will use current
                                   #batch stastics but during testing it uses learned running statics
    running_loss = 0

    for images, labels in train_loader: # loads images and labels from one batch
        optimizer.zero_grad()           # this will clear old gradient befor new one.like we are calcualting gradient for one batch and cleraing it.
                                        # we cal greadient per batch cause we are calcluating loss per every batch.

        outputs = model(images)         #passes the batch to thr module
        loss = criterion(outputs, labels)# loss in claculated for one batch
                                          ##calculate gradient based on the batch

       
        loss.backward()              #computs the gradients
        optimizer.step()              # this will use gradient to update weights and bias

        running_loss += loss.item()  # adds every batch loss

    print(f"Epoch {epoch+1}, Loss: {running_loss / len(train_loader):.4f}")

Epoch 1, Loss: 1.1162
Epoch 2, Loss: 0.8437
Epoch 3, Loss: 0.6342
Epoch 4, Loss: 0.5375
Epoch 5, Loss: 0.4564
Epoch 6, Loss: 0.3866
Epoch 7, Loss: 0.3418
Epoch 8, Loss: 0.2713
Epoch 9, Loss: 0.2188
Epoch 10, Loss: 0.1684


In [71]:
model.eval()                   #this will set model to testing mode
correct = 0                    # corrct predections
total = 0                      #totla number of predections

with torch.no_grad():          #turns gradient calculation during testing
    for images, labels in test_loader:
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)# this will return max score and its class,like we only wnat class so we ignore score

        total += labels.size(0)   # batch wise number of images are updated
        correct += (predicted == labels).sum().item() # will compare predicted and actuall and add the total correct prdection .

accuracy = 100 * correct / total
print(f"Test Accuracy: {accuracy:.2f}%")

Test Accuracy: 89.44%
